# Notebook 05 — Pre-Match Modeling Dataset

## Objective

This notebook constructs the final one-row-per-match dataset used by the
pre-match classification and regression experiments.

It combines:

1. fixture identity and chronology;
2. the validated conventional pre-match features from Notebook 01;
3. the leakage-safe historical multilayer-network features from Notebook 04;
4. the de-vigged bookmaker probabilities retained as an external benchmark;
5. two supervised-learning targets:
   - home/draw/away classification;
   - signed goal margin clipped to \([-5,+5]\).

No predictive model is fitted in this notebook.

The final table will be consumed unchanged by:

- Notebook 06 — baseline models;
- Notebook 07 — FIGS and network-feature evaluation.

A strict leakage audit is performed before export.

In [1]:
# -------------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# -------------------------------------------------------------------------
# DISPLAY SETTINGS
# -------------------------------------------------------------------------

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 220)
pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}",
)

RANDOM_STATE = 42

print("Notebook 05 environment configured.")

Notebook 05 environment configured.


In [2]:
# -------------------------------------------------------------------------
# INPUT PATHS
# -------------------------------------------------------------------------

MATCHES_PATH = Path(
    "../data/processed/"
    "integrated_matches_laliga_2015_2016.parquet"
)

NETWORK_FEATURES_PATH = Path(
    "../data/processed/"
    "prematch_match_network_features_laliga_2015_2016.parquet"
)


# -------------------------------------------------------------------------
# OUTPUT PATHS
# -------------------------------------------------------------------------

MODELING_DATASET_OUTPUT_PATH = Path(
    "../data/processed/"
    "prematch_modeling_dataset_laliga_2015_2016.parquet"
)

FEATURE_MANIFEST_OUTPUT_PATH = Path(
    "../data/processed/"
    "prematch_modeling_feature_manifest_laliga_2015_2016.csv"
)


MODELING_DATASET_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

FEATURE_MANIFEST_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# INPUT VALIDATION
# -------------------------------------------------------------------------

for required_path in [
    MATCHES_PATH,
    NETWORK_FEATURES_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input file not found:\n"
            f"{required_path.resolve()}"
        )

print("Notebook 05 paths configured.")
print(f"Matches : {MATCHES_PATH.resolve()}")
print(f"Network : {NETWORK_FEATURES_PATH.resolve()}")
print(f"Output  : {MODELING_DATASET_OUTPUT_PATH.resolve()}")

Notebook 05 paths configured.
Matches : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/integrated_matches_laliga_2015_2016.parquet
Network : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_match_network_features_laliga_2015_2016.parquet
Output  : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_modeling_dataset_laliga_2015_2016.parquet


In [3]:
# -------------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------------

matches_df = pd.read_parquet(
    MATCHES_PATH
)

network_features_df = pd.read_parquet(
    NETWORK_FEATURES_PATH
)


# -------------------------------------------------------------------------
# DATE NORMALIZATION
# -------------------------------------------------------------------------

if "clean_date" not in matches_df.columns:
    raise ValueError(
        "Integrated match table is missing clean_date."
    )

matches_df["clean_date"] = pd.to_datetime(
    matches_df["clean_date"],
    errors="raise",
)

network_features_df["match_date"] = pd.to_datetime(
    network_features_df["match_date"],
    errors="raise",
)


matches_df = (
    matches_df
    .sort_values(
        [
            "clean_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)

network_features_df = (
    network_features_df
    .sort_values(
        [
            "match_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)


print("Notebook 05 inputs loaded.")
print(
    f"Integrated matches : "
    f"{len(matches_df):,}"
)
print(
    f"Network rows       : "
    f"{len(network_features_df):,}"
)

Notebook 05 inputs loaded.
Integrated matches : 380
Network rows       : 380


In [4]:
# -------------------------------------------------------------------------
# SOURCE STRUCTURAL VALIDATION
# -------------------------------------------------------------------------

required_match_columns = {
    "match_id",
    "clean_date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
}

missing_match_columns = (
    required_match_columns
    - set(matches_df.columns)
)

if missing_match_columns:
    raise ValueError(
        "Missing required integrated-match columns:\n"
        f"{sorted(missing_match_columns)}"
    )


required_network_columns = {
    "match_id",
    "match_date",
    "home_team",
    "away_team",
    "home_prior_matches_available",
    "away_prior_matches_available",
}

missing_network_columns = (
    required_network_columns
    - set(network_features_df.columns)
)

if missing_network_columns:
    raise ValueError(
        "Missing required Notebook 04 columns:\n"
        f"{sorted(missing_network_columns)}"
    )


assert len(matches_df) == 380
assert len(network_features_df) == 380

assert matches_df["match_id"].is_unique
assert network_features_df["match_id"].is_unique

assert (
    matches_df["match_id"].notna().all()
)

assert (
    network_features_df["match_id"]
    .notna()
    .all()
)

assert set(
    matches_df["match_id"]
) == set(
    network_features_df["match_id"]
)

print("Source structural validation passed.")

Source structural validation passed.


In [5]:
# -------------------------------------------------------------------------
# SOURCE COLUMN INVENTORY
# -------------------------------------------------------------------------

match_column_inventory_df = pd.DataFrame(
    {
        "column": matches_df.columns,
        "dtype": [
            str(dtype)
            for dtype in matches_df.dtypes
        ],
        "missing_count": [
            int(
                matches_df[column]
                .isna()
                .sum()
            )
            for column in matches_df.columns
        ],
    }
)

network_column_inventory_df = pd.DataFrame(
    {
        "column": network_features_df.columns,
        "dtype": [
            str(dtype)
            for dtype
            in network_features_df.dtypes
        ],
        "missing_count": [
            int(
                network_features_df[column]
                .isna()
                .sum()
            )
            for column
            in network_features_df.columns
        ],
    }
)


print("Integrated-match columns:")
display(match_column_inventory_df)

print("\nNotebook 04 network columns:")
display(network_column_inventory_df)

Integrated-match columns:


,column,dtype,missing_count
0,match_id,int64,0
1,clean_date,datetime64[ns],0
2,home_team,string,0
3,away_team,string,0
4,home_score,int64,0
5,away_score,int64,0
6,implied_H,float64,0
7,implied_D,float64,0
8,implied_A,float64,0
9,bookie_margin,float64,0



Notebook 04 network columns:


,column,dtype,missing_count
0,match_id,int64,0
1,match_date,datetime64[ns],0
2,home_team,object,0
3,away_team,object,0
4,home_team_match_number,int16,0
...,...,...,...
184,diff_maximum_zone_recovery_expanding_mean,float64,10
185,diff_switching_factor_expanding_mean,float64,10
186,diff_finite_switching_zones_expanding_mean,float64,10
187,diff_undefined_switching_zones_expanding_mean,float64,10


In [7]:
# -------------------------------------------------------------------------
# RECONSTRUCT NOTEBOOK 01 CONVENTIONAL PRE-MATCH FEATURES
# -------------------------------------------------------------------------
#
# Notebook 01 created these features dynamically but did not persist them
# in integrated_matches_laliga_2015_2016.parquet.
#
# Exact historical definition:
#
# - previous five matches;
# - regardless of whether the team played home or away;
# - only matches strictly before the target fixture;
# - GF and GA are interpreted from the selected team's perspective;
# - opening fixtures receive GF=0.0, GA=0.0 and history_count=0;
# - history_count is capped at the rolling-window size of 5.
#
# This reproduces the Notebook 01 feature definition while keeping the
# final modeling table self-contained and reproducible.
# -------------------------------------------------------------------------

CONVENTIONAL_ROLLING_WINDOW = 5


# -------------------------------------------------------------------------
# BUILD TEAM-MATCH LONG TABLE
# -------------------------------------------------------------------------

home_team_results_df = (
    matches_df[
        [
            "match_id",
            "clean_date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
        ]
    ]
    .rename(
        columns={
            "home_team": "team",
            "away_team": "opponent",
            "home_score": "goals_for",
            "away_score": "goals_against",
        }
    )
    .assign(
        venue="home"
    )
)


away_team_results_df = (
    matches_df[
        [
            "match_id",
            "clean_date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
        ]
    ]
    .rename(
        columns={
            "away_team": "team",
            "home_team": "opponent",
            "away_score": "goals_for",
            "home_score": "goals_against",
        }
    )
    .assign(
        venue="away"
    )
)


team_results_long_df = pd.concat(
    [
        home_team_results_df,
        away_team_results_df,
    ],
    ignore_index=True,
)


team_results_long_df = (
    team_results_long_df
    .sort_values(
        [
            "team",
            "clean_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# STRICT HISTORICAL SHIFT
# -------------------------------------------------------------------------

team_results_long_df[
    "prior_goals_for"
] = (
    team_results_long_df
    .groupby(
        "team",
        sort=False,
    )["goals_for"]
    .shift(1)
)

team_results_long_df[
    "prior_goals_against"
] = (
    team_results_long_df
    .groupby(
        "team",
        sort=False,
    )["goals_against"]
    .shift(1)
)


# -------------------------------------------------------------------------
# PREVIOUS-FIVE GF / GA
# -------------------------------------------------------------------------

team_results_long_df[
    "rolling_gf"
] = (
    team_results_long_df
    .groupby(
        "team",
        sort=False,
    )["prior_goals_for"]
    .transform(
        lambda series: (
            series
            .rolling(
                window=CONVENTIONAL_ROLLING_WINDOW,
                min_periods=1,
            )
            .mean()
        )
    )
)

team_results_long_df[
    "rolling_ga"
] = (
    team_results_long_df
    .groupby(
        "team",
        sort=False,
    )["prior_goals_against"]
    .transform(
        lambda series: (
            series
            .rolling(
                window=CONVENTIONAL_ROLLING_WINDOW,
                min_periods=1,
            )
            .mean()
        )
    )
)


# -------------------------------------------------------------------------
# HISTORY COUNT
# -------------------------------------------------------------------------
#
# This reproduces len(recent_sample) from Notebook 01:
# 0, 1, 2, 3, 4, then 5 for the remainder of the season.
# -------------------------------------------------------------------------

team_results_long_df[
    "history_count"
] = (
    team_results_long_df
    .groupby(
        "team",
        sort=False,
    )
    .cumcount()
    .clip(
        upper=CONVENTIONAL_ROLLING_WINDOW
    )
    .astype("int8")
)


# Notebook 01 used zero-valued defaults for opening fixtures.
opening_fixture_mask = (
    team_results_long_df[
        "history_count"
    ].eq(0)
)

team_results_long_df.loc[
    opening_fixture_mask,
    [
        "rolling_gf",
        "rolling_ga",
    ],
] = 0.0


# -------------------------------------------------------------------------
# VALIDATE TEAM-LONG CONSTRUCTION
# -------------------------------------------------------------------------

assert len(
    team_results_long_df
) == 760

assert not team_results_long_df[
    [
        "match_id",
        "team",
    ]
].duplicated().any()

assert (
    team_results_long_df[
        "history_count"
    ]
    .between(
        0,
        5,
    )
    .all()
)

assert team_results_long_df.loc[
    opening_fixture_mask,
    "rolling_gf",
].eq(0.0).all()

assert team_results_long_df.loc[
    opening_fixture_mask,
    "rolling_ga",
].eq(0.0).all()


# Each of the 20 teams should have one opening fixture.
assert int(
    opening_fixture_mask.sum()
) == 20


print(
    "Conventional team-history features reconstructed."
)
print(
    f"Team-match rows: "
    f"{len(team_results_long_df):,}"
)
print(
    f"History-count range: "
    f"{team_results_long_df['history_count'].min()}–"
    f"{team_results_long_df['history_count'].max()}"
)

Conventional team-history features reconstructed.
Team-match rows: 760
History-count range: 0–5


In [8]:
# -------------------------------------------------------------------------
# HOME CONVENTIONAL FEATURES
# -------------------------------------------------------------------------

home_conventional_df = (
    team_results_long_df.loc[
        team_results_long_df[
            "venue"
        ].eq("home"),
        [
            "match_id",
            "team",
            "rolling_gf",
            "rolling_ga",
            "history_count",
        ],
    ]
    .rename(
        columns={
            "team": "home_team",
            "rolling_gf": (
                "feat_home_rolling_gf"
            ),
            "rolling_ga": (
                "feat_home_rolling_ga"
            ),
            "history_count": (
                "feat_home_history_count"
            ),
        }
    )
)


# -------------------------------------------------------------------------
# AWAY CONVENTIONAL FEATURES
# -------------------------------------------------------------------------

away_conventional_df = (
    team_results_long_df.loc[
        team_results_long_df[
            "venue"
        ].eq("away"),
        [
            "match_id",
            "team",
            "rolling_gf",
            "rolling_ga",
            "history_count",
        ],
    ]
    .rename(
        columns={
            "team": "away_team",
            "rolling_gf": (
                "feat_away_rolling_gf"
            ),
            "rolling_ga": (
                "feat_away_rolling_ga"
            ),
            "history_count": (
                "feat_away_history_count"
            ),
        }
    )
)


assert len(home_conventional_df) == 380
assert len(away_conventional_df) == 380

assert home_conventional_df[
    "match_id"
].is_unique

assert away_conventional_df[
    "match_id"
].is_unique


# -------------------------------------------------------------------------
# MERGE INTO MATCH TABLE
# -------------------------------------------------------------------------

matches_df = (
    matches_df
    .merge(
        home_conventional_df,
        on=[
            "match_id",
            "home_team",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        away_conventional_df,
        on=[
            "match_id",
            "away_team",
        ],
        how="left",
        validate="one_to_one",
    )
)


CONVENTIONAL_FEATURE_COLUMNS = [
    "feat_home_rolling_gf",
    "feat_home_rolling_ga",
    "feat_away_rolling_gf",
    "feat_away_rolling_ga",
    "feat_home_history_count",
    "feat_away_history_count",
]


assert matches_df[
    CONVENTIONAL_FEATURE_COLUMNS
].notna().all().all()


print(
    "Conventional pre-match features attached "
    "to the integrated match table."
)

display(
    matches_df[
        [
            "clean_date",
            "home_team",
            "away_team",
        ]
        + CONVENTIONAL_FEATURE_COLUMNS
    ]
    .head(15)
)

Conventional pre-match features attached to the integrated match table.


,clean_date,home_team,away_team,feat_home_rolling_gf,feat_home_rolling_ga,feat_away_rolling_gf,feat_away_rolling_ga,feat_home_history_count,feat_away_history_count
0,2015-08-21,Málaga,Sevilla,0.0000,0.0000,0.0000,0.0000,0,0
1,2015-08-22,Atlético Madrid,Las Palmas,0.0000,0.0000,0.0000,0.0000,0,0
2,2015-08-22,RC Deportivo La Coruña,Real Sociedad,0.0000,0.0000,0.0000,0.0000,0,0
3,2015-08-22,Espanyol,Getafe,0.0000,0.0000,0.0000,0.0000,0,0
4,2015-08-22,Rayo Vallecano,Valencia,0.0000,0.0000,0.0000,0.0000,0,0
5,2015-08-23,Athletic Club,Barcelona,0.0000,0.0000,0.0000,0.0000,0,0
6,2015-08-23,Sporting Gijón,Real Madrid,0.0000,0.0000,0.0000,0.0000,0,0
7,2015-08-23,Real Betis,Villarreal,0.0000,0.0000,0.0000,0.0000,0,0
8,2015-08-23,Levante UD,Celta Vigo,0.0000,0.0000,0.0000,0.0000,0,0
9,2015-08-24,Granada,Eibar,0.0000,0.0000,0.0000,0.0000,0,0


In [9]:
# -------------------------------------------------------------------------
# MANUAL CONVENTIONAL-FEATURE VALIDATION
# -------------------------------------------------------------------------

validation_team = (
    team_results_long_df[
        "team"
    ]
    .sort_values()
    .iloc[0]
)


validation_team_df = (
    team_results_long_df.loc[
        team_results_long_df[
            "team"
        ].eq(validation_team)
    ]
    .sort_values(
        [
            "clean_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)


validation_records = []

for row_index in range(
    min(
        10,
        len(validation_team_df),
    )
):

    current_row = (
        validation_team_df.iloc[
            row_index
        ]
    )

    historical_rows_df = (
        validation_team_df.iloc[
            max(
                0,
                row_index
                - CONVENTIONAL_ROLLING_WINDOW,
            ):
            row_index
        ]
    )

    expected_history_count = (
        len(historical_rows_df)
    )

    if expected_history_count == 0:
        expected_gf = 0.0
        expected_ga = 0.0

    else:
        expected_gf = (
            historical_rows_df[
                "goals_for"
            ].mean()
        )

        expected_ga = (
            historical_rows_df[
                "goals_against"
            ].mean()
        )


    assert (
        int(
            current_row[
                "history_count"
            ]
        )
        == expected_history_count
    )

    assert np.isclose(
        current_row[
            "rolling_gf"
        ],
        expected_gf,
        atol=1e-12,
    )

    assert np.isclose(
        current_row[
            "rolling_ga"
        ],
        expected_ga,
        atol=1e-12,
    )


    validation_records.append(
        {
            "team": validation_team,
            "match_id": int(
                current_row[
                    "match_id"
                ]
            ),
            "history_count": (
                expected_history_count
            ),
            "expected_gf": expected_gf,
            "stored_gf": (
                current_row[
                    "rolling_gf"
                ]
            ),
            "expected_ga": expected_ga,
            "stored_ga": (
                current_row[
                    "rolling_ga"
                ]
            ),
        }
    )


manual_conventional_validation_df = (
    pd.DataFrame(
        validation_records
    )
)


print(
    "Manual previous-five-match feature "
    "validation passed."
)

display(
    manual_conventional_validation_df
)

Manual previous-five-match feature validation passed.


,team,match_id,history_count,expected_gf,stored_gf,expected_ga,stored_ga
0,Athletic Club,266236,0,0.0000,0.0000,0.0000,0.0000
1,Athletic Club,3825578,1,0.0000,0.0000,1.0000,1.0000
2,Athletic Club,3825587,2,0.0000,0.0000,1.5000,1.5000
3,Athletic Club,3825595,3,1.0000,1.0000,1.3333,1.3333
4,Athletic Club,3825602,4,1.0000,1.0000,1.7500,1.7500
5,Athletic Club,3825609,5,1.0000,1.0000,1.8000,1.8000
6,Athletic Club,3825622,5,1.0000,1.0000,1.6000,1.6000
7,Athletic Club,3825630,5,1.6000,1.6000,1.4000,1.4000
8,Athletic Club,3825641,5,1.4000,1.4000,1.6000,1.6000
9,Athletic Club,3825650,5,1.8000,1.8000,1.0000,1.0000


In [10]:
# -------------------------------------------------------------------------
# VALIDATED CONVENTIONAL PRE-MATCH FEATURES
# -------------------------------------------------------------------------

CONVENTIONAL_FEATURE_COLUMNS = [
    "feat_home_rolling_gf",
    "feat_home_rolling_ga",
    "feat_away_rolling_gf",
    "feat_away_rolling_ga",
    "feat_home_history_count",
    "feat_away_history_count",
]


# -------------------------------------------------------------------------
# MARKET BENCHMARK COLUMNS
# -------------------------------------------------------------------------

MARKET_BASELINE_COLUMNS = [
    "market_prob_H",
    "market_prob_D",
    "market_prob_A",
]


missing_conventional_columns = [
    column
    for column in CONVENTIONAL_FEATURE_COLUMNS
    if column not in matches_df.columns
]

missing_market_columns = [
    column
    for column in MARKET_BASELINE_COLUMNS
    if column not in matches_df.columns
]


if missing_conventional_columns:
    raise ValueError(
        "Validated conventional features are missing:\n"
        f"{missing_conventional_columns}"
    )

if missing_market_columns:
    raise ValueError(
        "Market benchmark columns are missing:\n"
        f"{missing_market_columns}"
    )


assert matches_df[
    CONVENTIONAL_FEATURE_COLUMNS
].notna().all().all()

assert (
    matches_df[
        [
            "feat_home_history_count",
            "feat_away_history_count",
        ]
    ]
    .le(5)
    .all()
    .all()
)


print("Validated feature families found.")
print(
    f"Conventional features: "
    f"{len(CONVENTIONAL_FEATURE_COLUMNS)}"
)
print(
    f"Market benchmark cols : "
    f"{len(MARKET_BASELINE_COLUMNS)}"
)

Validated feature families found.
Conventional features: 6
Market benchmark cols : 3


In [11]:
# -------------------------------------------------------------------------
# CANONICAL TARGET CONSTRUCTION
# -------------------------------------------------------------------------

modeling_base_df = matches_df.copy()

raw_goal_margin = (
    modeling_base_df["home_score"]
    - modeling_base_df["away_score"]
)

modeling_base_df[
    "target_outcome_verified"
] = np.select(
    [
        raw_goal_margin > 0,
        raw_goal_margin < 0,
    ],
    [
        "H",
        "A",
    ],
    default="D",
)

modeling_base_df[
    "target_margin_verified"
] = (
    raw_goal_margin
    .clip(
        lower=-5,
        upper=5,
    )
    .astype("int8")
)


# -------------------------------------------------------------------------
# VERIFY AGAINST NOTEBOOK 01 TARGETS, WHEN PRESENT
# -------------------------------------------------------------------------

if "target_outcome" in modeling_base_df.columns:

    assert (
        modeling_base_df[
            "target_outcome"
        ].astype(str)
        == modeling_base_df[
            "target_outcome_verified"
        ].astype(str)
    ).all(), (
        "Notebook 01 target_outcome does not match "
        "the reconstructed H/D/A target."
    )

else:
    modeling_base_df[
        "target_outcome"
    ] = modeling_base_df[
        "target_outcome_verified"
    ]


if "target_margin" in modeling_base_df.columns:

    assert np.array_equal(
        modeling_base_df[
            "target_margin"
        ].astype(int),
        modeling_base_df[
            "target_margin_verified"
        ].astype(int),
    ), (
        "Notebook 01 target_margin does not match "
        "the reconstructed clipped margin."
    )

else:
    modeling_base_df[
        "target_margin"
    ] = modeling_base_df[
        "target_margin_verified"
    ]


modeling_base_df = (
    modeling_base_df
    .drop(
        columns=[
            "target_outcome_verified",
            "target_margin_verified",
        ]
    )
)


assert set(
    modeling_base_df[
        "target_outcome"
    ].unique()
) == {
    "H",
    "D",
    "A",
}

assert modeling_base_df[
    "target_margin"
].between(
    -5,
    5,
).all()

print("Classification and regression targets verified.")

Classification and regression targets verified.


In [12]:
# -------------------------------------------------------------------------
# TARGET DISTRIBUTIONS
# -------------------------------------------------------------------------

classification_distribution_df = (
    modeling_base_df[
        "target_outcome"
    ]
    .value_counts()
    .reindex(
        [
            "H",
            "D",
            "A",
        ]
    )
    .rename_axis(
        "target_outcome"
    )
    .reset_index(
        name="match_count"
    )
)

classification_distribution_df[
    "percentage"
] = (
    classification_distribution_df[
        "match_count"
    ]
    / len(modeling_base_df)
    * 100.0
)

print("H/D/A distribution:")
display(
    classification_distribution_df
)

print("\nClipped goal-margin distribution:")

display(
    modeling_base_df[
        "target_margin"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "target_margin"
    )
    .reset_index(
        name="match_count"
    )
)

H/D/A distribution:


,target_outcome,match_count,percentage
0,H,183,48.1579
1,D,92,24.2105
2,A,105,27.6316



Clipped goal-margin distribution:


,target_margin,match_count
0,-5,3
1,-4,7
2,-3,9
3,-2,32
4,-1,54
5,0,92
6,1,82
7,2,50
8,3,26
9,4,14


In [13]:
# -------------------------------------------------------------------------
# FAIR-MARKET PROBABILITY VALIDATION
# -------------------------------------------------------------------------

market_probability_matrix = (
    modeling_base_df[
        MARKET_BASELINE_COLUMNS
    ]
)

assert market_probability_matrix[
    MARKET_BASELINE_COLUMNS
].notna().all().all()

assert (
    market_probability_matrix >= 0
).all().all()

assert (
    market_probability_matrix <= 1
).all().all()

market_probability_sums = (
    market_probability_matrix.sum(
        axis=1
    )
)

assert np.allclose(
    market_probability_sums,
    1.0,
    atol=1e-6,
)

print(
    "De-vigged market-probability validation passed."
)

display(
    modeling_base_df[
        [
            "clean_date",
            "home_team",
            "away_team",
        ]
        + MARKET_BASELINE_COLUMNS
    ]
    .head(10)
)

De-vigged market-probability validation passed.


,clean_date,home_team,away_team,market_prob_H,market_prob_D,market_prob_A
0,2015-08-21,Málaga,Sevilla,0.2792,0.2792,0.4416
1,2015-08-22,Atlético Madrid,Las Palmas,0.7908,0.1460,0.0633
2,2015-08-22,RC Deportivo La Coruña,Real Sociedad,0.3825,0.2988,0.3187
3,2015-08-22,Espanyol,Getafe,0.4993,0.2805,0.2202
4,2015-08-22,Rayo Vallecano,Valencia,0.2642,0.2718,0.4640
5,2015-08-23,Athletic Club,Barcelona,0.1463,0.2196,0.6340
6,2015-08-23,Sporting Gijón,Real Madrid,0.0732,0.1465,0.7803
7,2015-08-23,Real Betis,Villarreal,0.3403,0.2932,0.3665
8,2015-08-23,Levante UD,Celta Vigo,0.2656,0.2897,0.4447
9,2015-08-24,Granada,Eibar,0.4992,0.2889,0.2119


In [14]:
# -------------------------------------------------------------------------
# NOTEBOOK 04 FEATURE IDENTIFICATION
# -------------------------------------------------------------------------

NETWORK_IDENTIFIER_COLUMNS = {
    "match_id",
    "match_date",
    "home_team",
    "away_team",
    "home_team_match_number",
    "away_team_match_number",
    "home_prior_matches_available",
    "away_prior_matches_available",
    "diff_prior_matches_available",
}

network_feature_columns = [
    column
    for column in network_features_df.columns
    if column not in NETWORK_IDENTIFIER_COLUMNS
]


# Only numeric network columns should enter modeling.
non_numeric_network_features = [
    column
    for column in network_feature_columns
    if not pd.api.types.is_numeric_dtype(
        network_features_df[column]
    )
]

if non_numeric_network_features:
    raise TypeError(
        "Unexpected non-numeric network predictors:\n"
        f"{non_numeric_network_features}"
    )


print(
    f"Notebook 04 network predictors identified: "
    f"{len(network_feature_columns):,}"
)

display(
    pd.DataFrame(
        {
            "network_feature": (
                network_feature_columns
            )
        }
    )
)

Notebook 04 network predictors identified: 180


,network_feature
0,home_completed_passes_roll3_mean
1,home_possession_losses_roll3_mean
2,home_possession_recoveries_roll3_mean
3,home_accumulated_eigenvector_centrality_roll3_...
4,home_maximum_zone_centrality_roll3_mean
...,...
175,diff_median_zone_recovery_expanding_mean
176,diff_maximum_zone_recovery_expanding_mean
177,diff_switching_factor_expanding_mean
178,diff_finite_switching_zones_expanding_mean


In [15]:
# -------------------------------------------------------------------------
# MERGE NETWORK FEATURES INTO MATCH TABLE
# -------------------------------------------------------------------------

network_merge_columns = [
    "match_id",
    "match_date",
    "home_team",
    "away_team",
    "home_team_match_number",
    "away_team_match_number",
    "home_prior_matches_available",
    "away_prior_matches_available",
    "diff_prior_matches_available",
] + network_feature_columns


final_modeling_df = (
    modeling_base_df
    .merge(
        network_features_df[
            network_merge_columns
        ],
        left_on=[
            "match_id",
            "clean_date",
            "home_team",
            "away_team",
        ],
        right_on=[
            "match_id",
            "match_date",
            "home_team",
            "away_team",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert len(final_modeling_df) == 380
assert final_modeling_df[
    "match_id"
].is_unique

assert final_modeling_df[
    "match_date"
].notna().all()

assert (
    final_modeling_df[
        "clean_date"
    ]
    == final_modeling_df[
        "match_date"
    ]
).all()

print("Notebook 01 and Notebook 04 tables merged.")
print(
    f"Rows   : "
    f"{len(final_modeling_df):,}"
)
print(
    f"Columns: "
    f"{final_modeling_df.shape[1]:,}"
)

Notebook 01 and Notebook 04 tables merged.
Rows   : 380
Columns: 207


In [16]:
# -------------------------------------------------------------------------
# CANONICAL COLUMN FAMILIES
# -------------------------------------------------------------------------

IDENTIFIER_COLUMNS = [
    "match_id",
    "clean_date",
    "home_team",
    "away_team",
]

TARGET_COLUMNS = [
    "target_outcome",
    "target_margin",
]

HISTORY_CONTEXT_COLUMNS = [
    "home_team_match_number",
    "away_team_match_number",
    "home_prior_matches_available",
    "away_prior_matches_available",
    "diff_prior_matches_available",
]


CANONICAL_MODEL_COLUMNS = (
    IDENTIFIER_COLUMNS
    + TARGET_COLUMNS
    + CONVENTIONAL_FEATURE_COLUMNS
    + MARKET_BASELINE_COLUMNS
    + HISTORY_CONTEXT_COLUMNS
    + network_feature_columns
)


missing_canonical_columns = [
    column
    for column in CANONICAL_MODEL_COLUMNS
    if column not in final_modeling_df.columns
]

if missing_canonical_columns:
    raise ValueError(
        "Canonical modeling columns missing:\n"
        f"{missing_canonical_columns}"
    )


prematch_modeling_df = (
    final_modeling_df[
        CANONICAL_MODEL_COLUMNS
    ]
    .sort_values(
        [
            "clean_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)


print("Canonical modeling table created.")
print(
    f"Rows   : "
    f"{len(prematch_modeling_df):,}"
)
print(
    f"Columns: "
    f"{prematch_modeling_df.shape[1]:,}"
)

Canonical modeling table created.
Rows   : 380
Columns: 200


In [17]:
# -------------------------------------------------------------------------
# EXPERIMENTAL FEATURE FAMILIES
# -------------------------------------------------------------------------

BASELINE_FEATURE_COLUMNS = (
    CONVENTIONAL_FEATURE_COLUMNS.copy()
)

NETWORK_ONLY_FEATURE_COLUMNS = (
    network_feature_columns.copy()
)

FULL_FEATURE_COLUMNS = (
    BASELINE_FEATURE_COLUMNS
    + NETWORK_ONLY_FEATURE_COLUMNS
)

MARKET_FEATURE_COLUMNS = (
    MARKET_BASELINE_COLUMNS.copy()
)


assert len(
    set(FULL_FEATURE_COLUMNS)
) == len(FULL_FEATURE_COLUMNS)

assert set(
    BASELINE_FEATURE_COLUMNS
).isdisjoint(
    NETWORK_ONLY_FEATURE_COLUMNS
)

assert set(
    MARKET_FEATURE_COLUMNS
).isdisjoint(
    FULL_FEATURE_COLUMNS
)


print("Feature families frozen for downstream notebooks.")
print(
    f"Conventional baseline : "
    f"{len(BASELINE_FEATURE_COLUMNS):,}"
)
print(
    f"Network only          : "
    f"{len(NETWORK_ONLY_FEATURE_COLUMNS):,}"
)
print(
    f"Combined              : "
    f"{len(FULL_FEATURE_COLUMNS):,}"
)
print(
    f"Market benchmark      : "
    f"{len(MARKET_FEATURE_COLUMNS):,}"
)

Feature families frozen for downstream notebooks.
Conventional baseline : 6
Network only          : 180
Combined              : 186
Market benchmark      : 3


In [18]:
# -------------------------------------------------------------------------
# FEATURE MANIFEST
# -------------------------------------------------------------------------

feature_manifest_records = []

for column in IDENTIFIER_COLUMNS:
    feature_manifest_records.append(
        {
            "column": column,
            "role": "identifier",
            "feature_family": "metadata",
            "used_as_predictor": False,
        }
    )

for column in TARGET_COLUMNS:
    feature_manifest_records.append(
        {
            "column": column,
            "role": "target",
            "feature_family": "target",
            "used_as_predictor": False,
        }
    )

for column in BASELINE_FEATURE_COLUMNS:
    feature_manifest_records.append(
        {
            "column": column,
            "role": "predictor",
            "feature_family": "conventional",
            "used_as_predictor": True,
        }
    )

for column in NETWORK_ONLY_FEATURE_COLUMNS:
    feature_manifest_records.append(
        {
            "column": column,
            "role": "predictor",
            "feature_family": "network",
            "used_as_predictor": True,
        }
    )

for column in MARKET_FEATURE_COLUMNS:
    feature_manifest_records.append(
        {
            "column": column,
            "role": "benchmark",
            "feature_family": "market",
            "used_as_predictor": False,
        }
    )

for column in HISTORY_CONTEXT_COLUMNS:
    feature_manifest_records.append(
        {
            "column": column,
            "role": "history_context",
            "feature_family": "metadata",
            "used_as_predictor": False,
        }
    )


feature_manifest_df = pd.DataFrame(
    feature_manifest_records
)

assert feature_manifest_df[
    "column"
].is_unique

print("Feature manifest constructed.")

display(
    feature_manifest_df[
        "feature_family"
    ]
    .value_counts()
    .rename_axis(
        "feature_family"
    )
    .reset_index(
        name="column_count"
    )
)

Feature manifest constructed.


,feature_family,column_count
0,network,180
1,metadata,9
2,conventional,6
3,market,3
4,target,2


In [19]:
# -------------------------------------------------------------------------
# MISSING-VALUE AUDIT
# -------------------------------------------------------------------------

predictor_columns = (
    FULL_FEATURE_COLUMNS
)

missing_predictor_summary_df = (
    prematch_modeling_df[
        predictor_columns
    ]
    .isna()
    .sum()
    .rename(
        "missing_count"
    )
    .to_frame()
)

missing_predictor_summary_df[
    "missing_percent"
] = (
    missing_predictor_summary_df[
        "missing_count"
    ]
    / len(prematch_modeling_df)
    * 100.0
)

missing_predictor_summary_df = (
    missing_predictor_summary_df.loc[
        missing_predictor_summary_df[
            "missing_count"
        ] > 0
    ]
    .sort_values(
        "missing_count",
        ascending=False,
    )
)


print(
    f"Predictor columns containing missing values: "
    f"{len(missing_predictor_summary_df):,}"
)

display(
    missing_predictor_summary_df
)


matches_with_any_missing_predictor = (
    prematch_modeling_df[
        predictor_columns
    ]
    .isna()
    .any(axis=1)
    .sum()
)

print(
    f"Matches with at least one missing predictor: "
    f"{matches_with_any_missing_predictor:,}"
)

Predictor columns containing missing values: 180


,missing_count,missing_percent
diff_undefined_switching_zones_expanding_mean,10,2.6316
home_completed_passes_roll3_mean,10,2.6316
home_possession_losses_roll3_mean,10,2.6316
home_possession_recoveries_roll3_mean,10,2.6316
diff_undefined_switching_zones_roll10_mean,10,2.6316
...,...,...
home_median_zone_leakage_roll3_mean,10,2.6316
home_maximum_zone_leakage_roll3_mean,10,2.6316
home_mean_zone_recovery_roll3_mean,10,2.6316
home_median_zone_recovery_roll3_mean,10,2.6316


Matches with at least one missing predictor: 10


In [20]:
# -------------------------------------------------------------------------
# STRICT LEAKAGE AUDIT
# -------------------------------------------------------------------------

FORBIDDEN_PREDICTOR_EXACT_NAMES = {
    "home_score",
    "away_score",
    "goal_difference",
    "goals_for",
    "goals_against",
    "target_outcome",
    "target_margin",
}


present_forbidden_predictors = (
    FORBIDDEN_PREDICTOR_EXACT_NAMES
    & set(FULL_FEATURE_COLUMNS)
)

assert not present_forbidden_predictors, (
    "Forbidden target/post-match columns appear in predictors:\n"
    f"{sorted(present_forbidden_predictors)}"
)


# -------------------------------------------------------------------------
# NETWORK FEATURE SAFETY
# -------------------------------------------------------------------------

for column in NETWORK_ONLY_FEATURE_COLUMNS:

    assert (
        "_roll" in column
        or "_expanding_" in column
    ), (
        "A network predictor does not appear to be historical: "
        f"{column}"
    )


# -------------------------------------------------------------------------
# MARKET SEPARATION
# -------------------------------------------------------------------------

assert set(
    MARKET_BASELINE_COLUMNS
).isdisjoint(
    FULL_FEATURE_COLUMNS
), (
    "Market probabilities must remain outside the "
    "primary model feature set."
)


# -------------------------------------------------------------------------
# TARGET COMPLETENESS
# -------------------------------------------------------------------------

assert prematch_modeling_df[
    TARGET_COLUMNS
].notna().all().all()

assert set(
    prematch_modeling_df[
        "target_outcome"
    ].unique()
) == {
    "H",
    "D",
    "A",
}

assert prematch_modeling_df[
    "target_margin"
].between(
    -5,
    5,
).all()


print("Strict pre-match leakage audit passed.")
print(
    "No current-match scores, outcomes, or "
    "unshifted network metrics are model predictors."
)

Strict pre-match leakage audit passed.
No current-match scores, outcomes, or unshifted network metrics are model predictors.


In [21]:
# -------------------------------------------------------------------------
# CHRONOLOGICAL ORDER VALIDATION
# -------------------------------------------------------------------------

assert prematch_modeling_df[
    "clean_date"
].is_monotonic_increasing

assert len(
    prematch_modeling_df
) == 380

assert (
    prematch_modeling_df[
        "match_id"
    ].nunique()
    == 380
)


date_differences = (
    prematch_modeling_df[
        "clean_date"
    ]
    .diff()
    .dropna()
)

assert (
    date_differences
    >= pd.Timedelta(0)
).all()


print("Chronological ordering validation passed.")
print(
    f"First match date: "
    f"{prematch_modeling_df['clean_date'].min().date()}"
)
print(
    f"Last match date : "
    f"{prematch_modeling_df['clean_date'].max().date()}"
)

Chronological ordering validation passed.
First match date: 2015-08-21
Last match date : 2016-05-15


In [22]:
# -------------------------------------------------------------------------
# NUMERIC PREDICTOR VALIDATION
# -------------------------------------------------------------------------

non_numeric_predictors = [
    column
    for column in FULL_FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(
        prematch_modeling_df[column]
    )
]

if non_numeric_predictors:
    raise TypeError(
        "Non-numeric predictor columns detected:\n"
        f"{non_numeric_predictors}"
    )


infinite_predictor_counts = {}

for column in FULL_FEATURE_COLUMNS:

    numeric_values = (
        prematch_modeling_df[column]
        .dropna()
        .to_numpy(
            dtype=np.float64
        )
    )

    infinite_count = int(
        np.isinf(
            numeric_values
        ).sum()
    )

    if infinite_count > 0:
        infinite_predictor_counts[
            column
        ] = infinite_count


assert not infinite_predictor_counts, (
    "Infinite predictor values detected:\n"
    f"{infinite_predictor_counts}"
)


print("Numeric predictor validation passed.")
print(
    f"Total model predictors: "
    f"{len(FULL_FEATURE_COLUMNS):,}"
)

Numeric predictor validation passed.
Total model predictors: 186


In [23]:
# -------------------------------------------------------------------------
# FINAL MODELING DATASET OVERVIEW
# -------------------------------------------------------------------------

dataset_summary_df = pd.DataFrame(
    {
        "item": [
            "Matches",
            "Conventional predictors",
            "Network predictors",
            "Combined predictors",
            "Market benchmark columns",
            "Classification targets",
            "Regression targets",
        ],
        "count": [
            len(prematch_modeling_df),
            len(
                BASELINE_FEATURE_COLUMNS
            ),
            len(
                NETWORK_ONLY_FEATURE_COLUMNS
            ),
            len(
                FULL_FEATURE_COLUMNS
            ),
            len(
                MARKET_BASELINE_COLUMNS
            ),
            prematch_modeling_df[
                "target_outcome"
            ].notna().sum(),
            prematch_modeling_df[
                "target_margin"
            ].notna().sum(),
        ],
    }
)

display(dataset_summary_df)


display(
    prematch_modeling_df[
        IDENTIFIER_COLUMNS
        + TARGET_COLUMNS
        + CONVENTIONAL_FEATURE_COLUMNS
        + MARKET_BASELINE_COLUMNS
    ]
    .head(10)
)

,item,count
0,Matches,380
1,Conventional predictors,6
2,Network predictors,180
3,Combined predictors,186
4,Market benchmark columns,3
5,Classification targets,380
6,Regression targets,380


,match_id,clean_date,home_team,away_team,target_outcome,target_margin,feat_home_rolling_gf,feat_home_rolling_ga,feat_away_rolling_gf,feat_away_rolling_ga,feat_home_history_count,feat_away_history_count,market_prob_H,market_prob_D,market_prob_A
0,3825562,2015-08-21,Málaga,Sevilla,D,0,0.0000,0.0000,0.0000,0.0000,0,0,0.2792,0.2792,0.4416
1,3825563,2015-08-22,Atlético Madrid,Las Palmas,H,1,0.0000,0.0000,0.0000,0.0000,0,0,0.7908,0.1460,0.0633
2,3825564,2015-08-22,RC Deportivo La Coruña,Real Sociedad,D,0,0.0000,0.0000,0.0000,0.0000,0,0,0.3825,0.2988,0.3187
3,3825565,2015-08-22,Espanyol,Getafe,H,1,0.0000,0.0000,0.0000,0.0000,0,0,0.4993,0.2805,0.2202
4,3825566,2015-08-22,Rayo Vallecano,Valencia,D,0,0.0000,0.0000,0.0000,0.0000,0,0,0.2642,0.2718,0.4640
5,266236,2015-08-23,Athletic Club,Barcelona,A,-1,0.0000,0.0000,0.0000,0.0000,0,0,0.1463,0.2196,0.6340
6,3825567,2015-08-23,Sporting Gijón,Real Madrid,D,0,0.0000,0.0000,0.0000,0.0000,0,0,0.0732,0.1465,0.7803
7,3825568,2015-08-23,Real Betis,Villarreal,D,0,0.0000,0.0000,0.0000,0.0000,0,0,0.3403,0.2932,0.3665
8,3825569,2015-08-23,Levante UD,Celta Vigo,A,-1,0.0000,0.0000,0.0000,0.0000,0,0,0.2656,0.2897,0.4447
9,3825570,2015-08-24,Granada,Eibar,A,-2,0.0000,0.0000,0.0000,0.0000,0,0,0.4992,0.2889,0.2119


In [24]:
# -------------------------------------------------------------------------
# EXPORT
# -------------------------------------------------------------------------

prematch_modeling_df.to_parquet(
    MODELING_DATASET_OUTPUT_PATH,
    index=False,
)

feature_manifest_df.to_csv(
    FEATURE_MANIFEST_OUTPUT_PATH,
    index=False,
)


print("Notebook 05 outputs exported.")

print(
    f"\nModeling dataset:\n"
    f"{MODELING_DATASET_OUTPUT_PATH.resolve()}"
)

print(
    f"\nFeature manifest:\n"
    f"{FEATURE_MANIFEST_OUTPUT_PATH.resolve()}"
)

print(
    f"\nRows: "
    f"{len(prematch_modeling_df):,}"
)

print(
    f"Columns: "
    f"{prematch_modeling_df.shape[1]:,}"
)

Notebook 05 outputs exported.

Modeling dataset:
/home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_modeling_dataset_laliga_2015_2016.parquet

Feature manifest:
/home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/prematch_modeling_feature_manifest_laliga_2015_2016.csv

Rows: 380
Columns: 200


In [25]:
# -------------------------------------------------------------------------
# RELOAD VALIDATION
# -------------------------------------------------------------------------

reloaded_modeling_df = pd.read_parquet(
    MODELING_DATASET_OUTPUT_PATH
)

reloaded_manifest_df = pd.read_csv(
    FEATURE_MANIFEST_OUTPUT_PATH
)


assert len(
    reloaded_modeling_df
) == 380

assert (
    reloaded_modeling_df[
        "match_id"
    ].nunique()
    == 380
)

assert not reloaded_modeling_df[
    "match_id"
].duplicated().any()

assert set(
    reloaded_modeling_df[
        "target_outcome"
    ].unique()
) == {
    "H",
    "D",
    "A",
}

assert reloaded_modeling_df[
    "target_margin"
].between(
    -5,
    5,
).all()

assert set(
    reloaded_manifest_df[
        "column"
    ]
) == set(
    feature_manifest_df[
        "column"
    ]
)


print("Notebook 05 reload validation passed.")
print(
    f"Reloaded matches: "
    f"{len(reloaded_modeling_df):,}"
)
print(
    f"Model predictors: "
    f"{len(FULL_FEATURE_COLUMNS):,}"
)
print(
    f"Market benchmark columns: "
    f"{len(MARKET_BASELINE_COLUMNS):,}"
)

Notebook 05 reload validation passed.
Reloaded matches: 380
Model predictors: 186
Market benchmark columns: 3


# Notebook 05 Summary — Pre-Match Modeling Dataset

## Objective

This notebook constructed the canonical one-row-per-match dataset for the
pre-match machine-learning experiments.

The dataset combines the validated conventional historical features from
Notebook 01 with the leakage-safe multilayer-network histories produced in
Notebook 04.

No predictive model was fitted.

---

## Prediction targets

Two supervised-learning targets are retained.

### Match-result classification

\[
Y_{\text{class}} \in \{H,D,A\}
\]

where:

- `H` = home win;
- `D` = draw;
- `A` = away win.

The class is derived from the original final-score difference.

### Signed goal-margin regression

\[
Y_{\text{margin}}
=
\operatorname{clip}
(
\text{home goals}
-
\text{away goals},
-5,
5
)
\]

The regression target therefore remains signed while limiting the influence
of rare extreme score margins.

---

## Conventional pre-match features

The validated conventional baseline consists of the six historical features
already established in Notebook 01:

```text
feat_home_rolling_gf
feat_home_rolling_ga
feat_away_rolling_gf
feat_away_rolling_ga
feat_home_history_count
feat_away_history_count